[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/04_%E6%AF%8D%E5%88%86%E6%95%A3%E3%83%BBF%E6%A4%9C%E5%AE%9A%E3%83%BB%E6%AF%8D%E6%AF%94%E7%8E%87%E3%81%AE%E5%B7%AE.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

# 統計2級-04. 母分散の検定・分散比(F)・母比率の差

**できるようになること**
- 母分散の検定（カイ二乗）ができる
- 2群の分散比(F)・母比率の差を検定できる
- 等分散かどうかでt検定を使い分けられる

**前提**：統計2級02・03　/　**所要**：約30分　/　**レベル**：統計検定2級相当

In [ ]:
import numpy as np               # 数値計算
import pandas as pd              # 表データ
import matplotlib.pyplot as plt  # グラフ描画
from scipy import stats          # 統計関数（分布・検定など）
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
rng = np.random.default_rng(0)   # 乱数生成器（seedで結果を固定）

## 1. 母分散の検定（カイ二乗分布）

「ばらつき（分散）が規定値と違うか」を検定。検定統計量は
$$ \chi^2 = \frac{(n-1)s^2}{\sigma_0^2} \sim \chi^2_{n-1} $$

例：製品の重さの分散は「4以下」が規格。標本から規格超過を疑えるか。

In [ ]:
data = np.array([50.5, 49.2, 51.8, 48.6, 52.3, 47.9, 50.1, 53.0, 48.0, 51.2])  # 重さの標本
n = len(data)           # 標本サイズ
s2 = data.var(ddof=1)   # 不偏分散
sigma0_2 = 4.0          # 帰無仮説の分散
chi2 = (n - 1) * s2 / sigma0_2   # 検定統計量（カイ二乗）
# 片側(分散が大きい方)の検定
p = 1 - stats.chi2.cdf(chi2, df=n - 1)   # 上側のp値
print(f'標本分散 s^2 = {s2:.2f}')
print(f'χ² = {chi2:.2f},  p値 = {p:.4f}')
print('結論:', '分散は4より大きい' if p < 0.05 else '規格内といえる')

## 2. 2つの母分散の比の検定（F検定）

2グループの**ばらつきが等しいか**を検定。$F = s_1^2 / s_2^2 \sim F_{n_1-1,\,n_2-1}$。
次の2標本t検定で「等分散かどうか」を判断する前段にもなります。

In [ ]:
groupA = rng.normal(50, 5, 15)             # A群（ばらつき小）
groupB = rng.normal(50, 9, 18)   # Bの方がばらつき大
F = groupA.var(ddof=1) / groupB.var(ddof=1)  # 分散比（F統計量）
dfA, dfB = len(groupA) - 1, len(groupB) - 1  # それぞれの自由度
# 両側p値
p = 2 * min(stats.f.cdf(F, dfA, dfB), 1 - stats.f.cdf(F, dfA, dfB))  # 両側のp値
print(f'F = {F:.3f},  p値 = {p:.4f}')
print('結論:', '分散は等しくない' if p < 0.05 else '等分散といえる')

## 3. 2標本t検定：等分散か否かで使い分け

- **等分散**を仮定 → Student のt検定（`equal_var=True`）
- **等分散を仮定しない** → Welch のt検定（`equal_var=False`、実務の既定）

In [ ]:
# 同じ2群を2通りで検定
print('Student (等分散仮定):', stats.ttest_ind(groupA, groupB, equal_var=True))   # 等分散を仮定
print('Welch  (非等分散)  :', stats.ttest_ind(groupA, groupB, equal_var=False))  # 等分散を仮定しない

## 4. 母比率の差の検定

2つのグループの「割合」に差があるか。A/Bテストの本質です。
例：施策前の申込率 vs 施策後の申込率。

In [ ]:
from statsmodels.stats.proportion import proportions_ztest  # 母比率の検定
count = np.array([84, 120])     # 申込数
nobs = np.array([1000, 1050])   # 訪問数
z, p = proportions_ztest(count, nobs)  # 2群の比率の差の検定
print(f'A: {count[0]/nobs[0]:.3f},  B: {count[1]/nobs[1]:.3f}')
print(f'z = {z:.3f},  p値 = {p:.4f}')
print('結論:', '比率に差あり' if p < 0.05 else '差は有意でない')

> **よくある間違い**：2群のt検定で「分散が等しい」と勝手に決めない。F検定で確認するか、迷ったら**Welch（等分散を仮定しない）**を既定にすると安全。

## 5. F分布の便利な性質：下側点は「入れ替えた上側点の逆数」

F分布表は**上側**しか載っていないことが多いです。下側の点は、自由度を入れ替えた上側点の逆数で求められます：`F(m,n)の下側α点 = 1 / F(n,m)の上側α点`。

In [ ]:
from scipy import stats
m, n, a = 10, 20, 0.05
lower = stats.f.ppf(a, m, n)              # F(10,20) の下側5%点
from_swap = 1 / stats.f.ppf(1-a, n, m)    # 1 / F(20,10) の上側5%点
print(f'F({m},{n}) 下側5%点      = {lower:.4f}')
print(f'1 / F({n},{m}) 上側5%点 = {from_swap:.4f}  → 一致')

> 等分散の検定（両側F検定）では、上側点と「下側点＝逆数」の両方を棄却限界に使います。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます（答えは表示されません）。

In [ ]:
# 採点用の関数を実行（答え合わせに使うだけ）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
import numpy as np
a=[2,4,6]; b=[1,2,3]
# Q1: 分散比 F = (aの不偏分散)/(bの不偏分散) を ans に
ans = None   # 例: np.var(a,ddof=1)/np.var(b,ddof=1)
_check('Q1 F', ans, np.var(a,ddof=1)/np.var(b,ddof=1))

In [ ]:
# Q2: 母分散の検定統計量 χ²=(n-1)s²/σ0²。n=10, s²=6, σ0²=4 のとき ans は？
ans = None   # 9*6/4
_check('Q2 χ²', ans, 9*6/4)

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
# Q3: 申込[84,120]/訪問[1000,1050] の母比率の差の z を ans に
ans = None   # 例: proportions_ztest([84,120],[1000,1050])[0]
_check('Q3 z', ans, proportions_ztest([84,120],[1000,1050])[0])

In [ ]:
from scipy import stats
# Q4: F(8,12) の下側2.5%点を、逆数性質 1/F(12,8)上側2.5%点 で求めて ans に
ans = None   # 例: 1/stats.f.ppf(0.975, 12, 8)
_check('Q4 F下側点', ans, 1/stats.f.ppf(0.975, 12, 8))

---
## 練習問題

**問1.** ある機械の加工誤差の分散は規格0.5。標本
`[10.2,9.7,10.5,9.4,10.8,9.6,10.1,10.6]` から、分散が規格より大きいか検定しよう。

**問2.** 2クラスのテスト点 A=`rng.normal(60,8,20)`, B=`rng.normal(60,15,20)` で、
分散が等しいかF検定し、その結果に応じてt検定（equal_varを選択）をしよう。

**問3.** 広告Aは2000人中160人、広告Bは1800人中180人がクリック。
クリック率に差があるか母比率の差の検定で調べよう。

**問4.** F(5,10) の下側5%点を、逆数性質を使って求めよう。`stats.f.ppf(0.05,5,10)` と一致する？

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[04_母分散・F検定・母比率の差 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/04_%E6%AF%8D%E5%88%86%E6%95%A3%E3%83%BBF%E6%A4%9C%E5%AE%9A%E3%83%BB%E6%AF%8D%E6%AF%94%E7%8E%87%E3%81%AE%E5%B7%AE.md)**

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 母分散の検定 | 分散が規定値かをχ²で |
| 分散比 F検定 | 2群の分散が等しいか |
| 等分散/Welch | 等分散仮定する/しないt検定 |
| 母比率の差 | 2群の割合の差の検定 |

In [ ]:
# チートシート（分散・比率）
from scipy import stats
stats.f.cdf(2.0, 5, 7)                          # F分布の確率
stats.ttest_ind([1,2,3],[2,3,4], equal_var=False)   # Welchのt検定
from statsmodels.stats.proportion import proportions_ztest
proportions_ztest([84,120],[1000,1050])         # 母比率の差

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""       # ← あなたの名前（例: 山田）
LOG_URL = ""    # ← 記録用URL（配布時に設定）
LOG_TOKEN = ""  # ← 記録用トークン（配布時に設定）
NOTEBOOK = "04_統計_2級/04_母分散・F検定・母比率の差"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")